# Getting Started

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CMBSciPol/jax-healpy/blob/main/docs/user-guide/getting-started.ipynb)

This page follows the steps of the [healpy tutorial](https://healpy.readthedocs.io/en/latest/tutorial.html)
with jax-healpy. jax-healpy, imported as `hp`, does the computations, with the names and the
arguments of healpy. healpy, imported under its own name, draws the maps and reads and writes
the files, which jax-healpy leaves to it.

In [ ]:
# Install jax-healpy and its dependencies on Google Colab. A CPU runtime, the default one, is expected:
# the `cpu` extra installs ducc0, which computes the spherical harmonic transforms on CPU only.
import sys

if 'google.colab' in sys.modules:
    %pip install --quiet "jax-healpy[cpu] @ git+https://github.com/CMBSciPol/jax-healpy" healpy
    !wget --quiet --directory-prefix=data https://raw.githubusercontent.com/CMBSciPol/jax-healpy/main/docs/user-guide/data/cmb_cls.txt

In [ ]:
import tempfile
from pathlib import Path

import healpy
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import jax_healpy as hp

jax.config.update('jax_enable_x64', True)

!!! note
    jax-healpy follows JAX's precision setting and does not enable 64-bit precision on import. With the
    default 32-bit precision, the results diverge from healpy even at moderate resolution (e.g. `nside = 256`).
    64-bit precision should be enabled before the first array operation.

## Resolution and pixel order

The resolution of a map is set by `nside`: the sphere is divided into $12\,n_\mathrm{side}^2$
pixels of equal area. The pixel functions of jax-healpy require `nside` to be a power of 2.

In [ ]:
nside = 32
print(f'Approximate resolution at nside = {nside}: {hp.nside2resol(nside, arcmin=True) / 60:.2f} deg')
npix = hp.nside2npix(nside)
print(f'Number of pixels: {npix}')

A map is an array of `npix` values. By default, the pixels are in the RING order: their
index increases along the rings of constant latitude, from the North pole to the South pole.
healpy's `mollview` draws a map in the Mollweide projection, here the index of each pixel:

In [ ]:
m = jnp.arange(npix)
healpy.mollview(m, title='Pixel index, RING order')
healpy.graticule()
plt.show()

A direction on the sphere is given either by two angles, the colatitude $\theta$, from 0 at the
North pole to $\pi$ at the South pole, and the longitude $\phi$, or by a unit vector.
`query_disc` finds the pixels within a radius of a direction, here 10° around a point of the
equator:

In [ ]:
vec = hp.ang2vec(np.pi / 2, np.pi * 3 / 4)
print(vec)
ipix_disc = hp.query_disc(nside, vec, radius=np.radians(10))

Two differences from healpy show when the disc is drawn on the map:

- A JAX array cannot be modified in place: `m.at[indices].set(values)` returns a modified copy.
- `query_disc` returns an array of fixed size, padded with the value `npix`, since the number
  of pixels in the disc is not known when the function is compiled. `mode='drop'` ignores the
  padding, whose indices are out of bounds.

[Migrating from healpy](migrating-from-healpy.md) lists the other differences.

In [ ]:
m = jnp.arange(npix).at[ipix_disc].set(npix - 1, mode='drop')
healpy.mollview(m, title='Pixels within 10° of (θ, φ) = (90°, 135°)')
plt.show()

`pix2ang` returns the angles of the centers of the pixels, in radians. The first four pixels
form the ring closest to the North pole:

In [ ]:
theta, phi = np.degrees(hp.pix2ang(nside, jnp.arange(5)))
print(f'theta: {theta}')
print(f'phi:   {phi}')

In the NESTED order, the indices follow the hierarchy of the pixels: each of the 12 base
pixels holds a block of consecutive indices. [HEALPix pixelization](pixelization.md) details
the two orders and the conversions between them.

In [ ]:
healpy.mollview(jnp.arange(npix), nest=True, title='Pixel index, NESTED order')
plt.show()

## Reading and writing maps

jax-healpy does not read or write files: healpy's `read_map` and `write_map` do, and the
functions of jax-healpy accept the NumPy arrays that `read_map` returns.

The healpy tutorial downloads a map of WMAP. This page simulates instead a map of the cosmic
microwave background (CMB), at `nside = 256`, from its temperature power spectrum: the file
[`data/cmb_cls.txt`](data/cmb_cls.txt) gives it in μK², computed with CAMB for the Planck 2018
cosmology (see [Polarization](polarization.md)). `synfast` takes a JAX PRNG key, where healpy
draws from the global random state of NumPy.

In [ ]:
nside = 256
lmax = 3 * nside - 1
ell, cl_tt = np.loadtxt('data/cmb_cls.txt', usecols=(0, 1))[: lmax + 1].T
cmb_map = hp.synfast(jax.random.key(0), cl_tt, nside)

with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / 'cmb_map.fits'
    healpy.write_map(path, cmb_map, dtype=np.float64)
    cmb_map_read = healpy.read_map(path)

print(f'Map read back from the file: {type(cmb_map_read).__name__}, identical: {np.array_equal(cmb_map_read, cmb_map)}')

## Visualization

healpy draws the maps of jax-healpy as its own. The Mollweide view can equalize the histogram
of the colors, which brings out the structures at all the scales:

In [ ]:
healpy.mollview(cmb_map, title='Simulated CMB, histogram-equalized colors', unit='μK', norm='hist')
healpy.graticule()
plt.show()

The gnomonic view zooms on a region, here a patch of about 17° centered at a longitude of 0°
and a latitude of 0.3°:

In [ ]:
healpy.gnomview(cmb_map, rot=(0, 0.3), reso=5, title='Gnomonic view', unit='μK', format='%.2g')
plt.show()

## Masked maps

The pixels without data hold the value `UNSEEN`, which jax-healpy defines as healpy does. Here,
a band of 20° on both sides of the equator is masked, as the Galactic plane is in the analyses
of the CMB:

In [ ]:
theta, _ = hp.pix2ang(nside, jnp.arange(hp.nside2npix(nside)))
mask = jnp.abs(np.pi / 2 - theta) < np.radians(20)
masked_map = jnp.where(mask, hp.UNSEEN, cmb_map)

healpy.mollview(masked_map, title='Masked map', unit='μK')
plt.show()

`bad_pixel_mask` finds the `UNSEEN` pixels, so that the others can be selected, here to draw
the histogram of their values:

In [ ]:
observed = ~hp.bad_pixel_mask(masked_map)
plt.hist(masked_map[observed], bins=100)
plt.xlabel('Temperature [μK]')
plt.ylabel('Number of pixels')
plt.show()

## Spherical harmonic transforms

The spherical harmonic transforms require the `cpu` or the `gpu` extra dependencies, see the
[installation](installation.md#spherical-harmonic-transforms).

`anafast` computes the angular power spectrum of a map. healpy sets the `UNSEEN` pixels to zero
before a transform, whereas jax-healpy never inspects the values of the pixels, so that the
transforms stay linear functions of the map: the masked pixels are set to zero explicitly. The
spectrum of the masked map is then lower than the spectrum of the full sky by about the fraction
of the sky that is kept, $f_\mathrm{sky}$: divided by it, it follows the input spectrum up to
$\ell \approx 2\,n_\mathrm{side}$. Beyond, jax-healpy and healpy both underestimate it by a few
percent, as the pixels sample these scales only approximately.

In [ ]:
cl = hp.anafast(jnp.where(mask, 0, cmb_map), lmax=lmax)
f_sky = 1 - jnp.mean(mask)

plt.figure(figsize=(8, 4))
plt.plot(ell, ell * (ell + 1) * cl / f_sky / (2 * np.pi), label=r'Masked map, divided by $f_\mathrm{sky}$')
plt.plot(ell, ell * (ell + 1) * cl_tt / (2 * np.pi), label='Input spectrum')
plt.xlabel(r'$\ell$')
plt.ylabel(r'$\ell(\ell+1)C_\ell/2\pi$ [μK²]')
plt.legend()
plt.grid()
plt.show()

healpy's `write_cl` writes a spectrum to a file. `smoothing` convolves a map with a Gaussian
beam, here of 1° of full width at half maximum. The width of the beam sets the computation
that is compiled: pass a Python or a NumPy float, not a JAX array.

In [ ]:
smoothed_map = hp.smoothing(cmb_map, fwhm=np.radians(1))
healpy.mollview(smoothed_map, title='Map smoothed with a 1° beam', unit='μK')
plt.show()

## What JAX adds

Up to here, the code is that of healpy, with another import. The functions of jax-healpy also
compose with the transformations of JAX:

- `jax.jit` compiles a function, for the CPU or the GPU;
- `jax.vmap` maps it over a batch, here of 8 simulations;
- `jax.grad` differentiates it, here the mean power of a simulated map with respect to the
  amplitude of its spectrum. The power is proportional to the amplitude, so the derivative at
  an amplitude of 1 is the power itself.

In [ ]:
@jax.jit
def mean_power(amplitude, key):
    simulated_map = hp.synfast(key, amplitude * cl_tt, nside)
    return jnp.mean(simulated_map**2)


keys = jax.random.split(jax.random.key(1), 8)
powers = jax.vmap(mean_power, in_axes=(None, 0))(1.0, keys)
derivatives = jax.vmap(jax.grad(mean_power), in_axes=(None, 0))(1.0, keys)

print(f'Mean powers of 8 realizations [μK²]: {np.round(powers)}')
print(f'Derivatives [μK²]:                   {np.round(derivatives)}')

[Performance tips](performance-tips.md) develops the compilation, the batching and the use of
the GPU. [Polarization](polarization.md) transforms the polarized maps, and
[Inference of the tensor-to-scalar ratio](inference.md) differentiates a likelihood of the
polarized spectra to sample it with NumPyro.